In [ ]:
%%sql
delete from lh_vv_bronze.dbo.brz_watermark where watermark_id = 'POI_CHANGE_SYNC';
delete from lh_vv_bronze.dbo.poi_change_entity_version;
delete from lh_vv_bronze.dbo.poi_change_event_rejected;

StatementMeta(, c6ee3178-4491-4e47-8432-52885e1ee7a0, 9, Finished, Available, Finished, True)

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 1 rows and 1 fields>

In [1]:
# Fabric Notebook: NB_POI_CHANGE_EVENT_PROCESSOR_V11_KEY_LEVEL_HISTORY
# V11 FINAL DESIGN
# - No poi_change_entity_version dependency.
# - No Gold schema changes.
# - Durable ordering is resolved from Bronze history per
#   (poi_id, entity, change_key).
# - For duplicate keys across versions: highest version wins; event_id breaks ties.
# - Keys absent from newer events are preserved from the latest event that contains them.
# - Explicit JSON null is a real patch when the target column is nullable.
# - Historical resolution is restricted to POIs/keys affected by the current batch.
#
# =============================================================================
# V11 KEY-LEVEL HISTORY RESOLUTION + PERFORMANCE OBSERVABILITY
#   Goal: low-latency processing for small incremental POI change batches.
#
#   Durable safety retained:
#     - fixed ingest_timestamp watermark window + lookback
#     - eventId idempotency
#     - basic event / JSON validation
#     - key-level historical ordering from Bronze event history
#     - latest value per (poi_id, entity, patch_key)
#     - Gold + ai_enriched_poi mutation must succeed before watermark advance
# #     - brz_pipeline_run run-level observability
#
#   V10.3 additions:
#     - INITIAL_WATERMARK remains 2026-09-13 00:00:00 for bootstrap.
#     - Retry optimistic Delta concurrency for Gold, AI, event-control,
#       and watermark mutations.
#     - AI retry rebuilds the current Gold-derived source on every attempt,
#       rather than blindly retrying a stale source snapshot.
#     - Retry remains bounded; non-concurrency errors fail immediately.
#
#   Hot-path work intentionally removed/simplified:
#     - NO Shared STG / COPY STG
#     - NO full-Bronze NULL ingest_timestamp anti-join scan
#     - NO duplicate-event payload conflict aggregation
#     - NO same-(poi,entity,version) payload conflict aggregation
#     - NO pre-MERGE target-existence join for each Gold entity
#     - NO per-column Window/Join patch plan
#     - NO runtime duplicate DQ scan of ai_enriched_poi source
#     - NO post-MERGE AI missing-row anti-join
#     - ONE event-control MERGE for processed/rejected/stale
#
#   Contract assumption:
#     - upstream eventId is stable/unique for one logical event
#     - NB00-NB60 owns full AI materialization; Change Flow updates existing AI rows
# =============================================================================
# V10.8 RUNTIME CONTROL MODEL
#   brz_watermark              -> incremental cursor
##   brz_pipeline_run           -> run-level audit/performance
#   persistent event-level processed table -> not used
#   poi_change_event_rejected -> rejected-event DLQ/audit; changes=original Bronze JSON, error_reason=validation _error
#
# V10.8 FINAL SUCCESS SIGNAL
#   The batch is successful only after these logs complete in order:
#     REJECT_DLQ_DONE
#     KEY_LEVEL_ORDERING_DONE
#     ACCOUNTING
#     WATERMARK_DONE
#     PIPELINE_RUN_DONE
#     COMPLETE status=SUCCESS
#
#   WATERMARK_DONE means brz_watermark.watermark_value has been advanced to
#   this batch's fixed watermark_to.
#
# V10.8 PERFORMANCE OBSERVABILITY
#   Business rules are preserved from V10.7.
#   Added true step_seconds timing around expensive Delta operations:
#     REJECT_DLQ, each Gold entity MERGE, AI_SYNC, KEY_LEVEL_ORDERING, WATERMARK.
#   This identifies the exact table/action responsible for long runtimes.
#
# =============================================================================
# PURPOSE
#   Incrementally process brz_poi_changes_events and apply validated changes to
#   the current Gold serving state. This notebook is the runtime processor for
#   POI change events; it does NOT write Shared STG / COPY STG.
#
#       Bronze change events
#            -> freeze watermark window
#            -> validate / deduplicate
#            -> resolve latest value per affected key from Bronze history
#            -> patch current Gold POI state
#            -> incrementally refresh existing ai_enriched_poi rows
#            -> finalize event control state
#            -> advance brz_watermark
#
# PERFORMANCE / OBSERVABILITY
#   - Cache the fixed raw window because it is reused by multiple actions.
#   - Cache validated/ready datasets to avoid rebuilding the validation DAG for
#     repeated counts, Gold patching and AI sync.
#   - Compute event accounting counts in one aggregation instead of 4 full scans.
#   - Emit stage logs with event counts and elapsed time.
#   - Reuse brz_pipeline_run for run status, raw_event_count, start/end timestamps,
#     duration (when the physical column type is supported) and error_message.
#   - Always unpersist cached datasets at the end of the run.
#
# IMPORTANT
#   - Handles: poi_master, poi_detail, poi_enrichment_display.
#   - PostgreSQL staging/sync is intentionally outside this notebook.
#   - Pipeline concurrency MUST be 1.
#
# V4 WATERMARK CONTRACT (actual Fabric schema)
#   watermark_id
#   table_name
#   watermark_column
#   watermark_value
#   update_time
#   status
#   flow_name
#   watermark_sequence
#   last_run_id
#   error_message
#
# SAFETY
#   - ingest_timestamp is the incremental cursor.
#   - fixed upper bound per run.
#   - 5 minute lookback.
#   - event_id idempotency.
#   - key-level durable ordering from Bronze history.
#   - conflicting duplicate event detection.
#   - invalid casts rejected before Gold mutation.
#   - watermark advances ONLY after Gold + ai_enriched_poi succeed.
#   - Change processor pipeline concurrency MUST be 1.
# =============================================================================

import json
import uuid
import time
from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql.window import Window


# =============================================================================
# 0. CONFIG
# =============================================================================

BRONZE_DB = "sgr_visitvn_dev.lh_vv_bronze.dbo"
GOLD_DB = "sgr_visitvn_dev.lh_vv_gold.dbo"

BRONZE_EVENT_TBL = f"{BRONZE_DB}.brz_poi_changes_events"

REJECTED_TBL = "lh_vv_bronze.dbo.poi_change_event_rejected"
WATERMARK_TBL = f"{BRONZE_DB}.brz_watermark"
PIPELINE_RUN_TBL = f"{BRONZE_DB}.brz_pipeline_run"

FLOW_NAME = "POI_CHANGE_PROCESS"
NOTEBOOK_NAME = "NB_POI_CHANGE_EVENT_PROCESSOR_V11_KEY_LEVEL_HISTORY"
RUN_STARTED_MONOTONIC = time.perf_counter()

WATERMARK_ID = "POI_CHANGE_SYNC"
WATERMARK_TABLE_NAME = "brz_poi_changes_events"
WATERMARK_SOURCE_COLUMN = "ingest_timestamp"
INITIAL_WATERMARK = "2026-09-13 00:00:00"
LOOKBACK_MINUTES = 5

RUN_ID = str(uuid.uuid4())

ENTITY_CONFIG = {
    "poi_master": {
        "target": f"{GOLD_DB}.poi_master",
        "sync_key": "poi_id",
    },
    "poi_detail": {
        "target": f"{GOLD_DB}.poi_detail",
        "sync_key": "poi_id",
    },
    "poi_enrichment_display": {
        "target": f"{GOLD_DB}.poi_enrichment_display",
        "sync_key": "poi_id",
    },
}

SUPPORTED_EVENT_TYPES = {
    "CREATE", "CREATED", "UPDATE", "UPDATED",
    "UPSERT", "PATCH", "POI_UPDATED", "POI_UPDATE"
}

PROTECTED_PATCH_COLUMNS = {
    "poi_id",
    "created_at",
}

REQUIRED_BRONZE_COLUMNS = {
    "actor",
    "poiId",
    "entity",
    "source",
    "changes",
    "eventId",
    "version",
    "eventType",
    "occurredAt",
    "schemaVersion",
    "ingest_timestamp",
}

REQUIRED_WATERMARK_COLUMNS = {
    "watermark_id",
    "table_name",
    "watermark_column",
    "watermark_value",
    "update_time",
    "status",
    "flow_name",
    "watermark_sequence",
    "last_run_id",
    "error_message",
}





# =============================================================================
# 1. HELPERS
# =============================================================================

def qcol(name):
    return f"`{name.replace('`', '``')}`"


def sql_string(value):
    if value is None:
        return "NULL"
    return "'" + str(value).replace("'", "''") + "'"


def table_exists(name):
    try:
        _ = spark.table(name).schema
        return True
    except Exception:
        return False


def columns(name):
    return list(spark.table(name).columns)


def column_set(name):
    return set(columns(name))


def validate_columns(name, required, label):
    missing = sorted(set(required) - column_set(name))
    if missing:
        raise RuntimeError(
            f"{label}_SCHEMA_MISMATCH table={name} missing={missing}"
        )


def row_count(name):
    return int(spark.sql(f"SELECT COUNT(*) c FROM {name}").first()["c"])


def duplicate_key_groups(name, key):
    return int(spark.sql(f"""
        SELECT COUNT(*) c
        FROM (
            SELECT {qcol(key)}
            FROM {name}
            WHERE {qcol(key)} IS NOT NULL
            GROUP BY {qcol(key)}
            HAVING COUNT(*) > 1
        ) x
    """).first()["c"])


def pipeline_run_columns():
    return column_set(PIPELINE_RUN_TBL)


def insert_pipeline_run(wm_from, wm_to, raw_count):
    cols = pipeline_run_columns()

    values = {
        "run_id": sql_string(RUN_ID),
        "flow_name": sql_string(FLOW_NAME),
        "status": sql_string("RUNNING"),
        "watermark_from": (
            f"CAST({sql_string(wm_from)} AS TIMESTAMP)"
            if "watermark_from" in cols else "NULL"
        ),
        "watermark_to": (
            f"CAST({sql_string(wm_to)} AS TIMESTAMP)"
            if "watermark_to" in cols else "NULL"
        ),
        "raw_event_count": str(int(raw_count)),
        "current_notebook": sql_string(NOTEBOOK_NAME),
        "started_at": "current_timestamp()",
        "ended_at": "NULL",
        "duration": "NULL",
        "error_message": "NULL",
    }

    insert_cols = [c for c in values if c in cols]

    spark.sql(f"""
        INSERT INTO {PIPELINE_RUN_TBL}
        ({", ".join(qcol(c) for c in insert_cols)})
        VALUES
        ({", ".join(values[c] for c in insert_cols)})
    """)


def update_pipeline_run(status, error_message=None, close_run=False):
    if not table_exists(PIPELINE_RUN_TBL):
        return

    cols = pipeline_run_columns()
    assignments = []

    if "status" in cols:
        assignments.append(f"status = {sql_string(status)}")

    if "current_notebook" in cols:
        assignments.append(
            f"current_notebook = {sql_string(NOTEBOOK_NAME)}"
        )

    if "error_message" in cols:
        msg = str(error_message)[:4000] if error_message else None
        assignments.append(f"error_message = {sql_string(msg)}")

    if close_run and "ended_at" in cols:
        assignments.append("ended_at = current_timestamp()")

    if assignments:
        spark.sql(f"""
            UPDATE {PIPELINE_RUN_TBL}
            SET {", ".join(assignments)}
            WHERE run_id = {sql_string(RUN_ID)}
        """)


def watermark_row():
    return spark.sql(f"""
        SELECT *
        FROM {WATERMARK_TBL}
        WHERE watermark_id = {sql_string(WATERMARK_ID)}
        LIMIT 1
    """).first()


def bootstrap_watermark():
    """
    Create the flow watermark only if it does not exist.
    Uses actual brz_watermark columns.
    """
    spark.sql(f"""
        MERGE INTO {WATERMARK_TBL} t
        USING (
            SELECT
                {sql_string(WATERMARK_ID)} AS watermark_id,
                {sql_string(WATERMARK_TABLE_NAME)} AS table_name,
                {sql_string(WATERMARK_SOURCE_COLUMN)} AS watermark_column,
                CAST({sql_string(INITIAL_WATERMARK)} AS TIMESTAMP) AS watermark_value,
                current_timestamp() AS update_time,
                'SUCCESS' AS status,
                {sql_string(FLOW_NAME)} AS flow_name,
                CAST(0 AS BIGINT) AS watermark_sequence,
                CAST(NULL AS STRING) AS last_run_id,
                CAST(NULL AS STRING) AS error_message
        ) s
        ON t.watermark_id = s.watermark_id

        WHEN NOT MATCHED THEN INSERT (
            watermark_id,
            table_name,
            watermark_column,
            watermark_value,
            update_time,
            status,
            flow_name,
            watermark_sequence,
            last_run_id,
            error_message
        )
        VALUES (
            s.watermark_id,
            s.table_name,
            s.watermark_column,
            s.watermark_value,
            s.update_time,
            s.status,
            s.flow_name,
            s.watermark_sequence,
            s.last_run_id,
            s.error_message
        )
    """)


def set_watermark_running():
    spark.sql(f"""
        UPDATE {WATERMARK_TBL}
        SET
            table_name = {sql_string(WATERMARK_TABLE_NAME)},
            watermark_column = {sql_string(WATERMARK_SOURCE_COLUMN)},
            update_time = current_timestamp(),
            status = 'RUNNING',
            flow_name = {sql_string(FLOW_NAME)},
            last_run_id = {sql_string(RUN_ID)},
            error_message = NULL
        WHERE watermark_id = {sql_string(WATERMARK_ID)}
    """)


def advance_watermark(wm_to):
    spark.sql(f"""
        UPDATE {WATERMARK_TBL}
        SET
            table_name = {sql_string(WATERMARK_TABLE_NAME)},
            watermark_column = {sql_string(WATERMARK_SOURCE_COLUMN)},
            watermark_value = CAST({sql_string(wm_to)} AS TIMESTAMP),
            update_time = current_timestamp(),
            status = 'SUCCESS',
            flow_name = {sql_string(FLOW_NAME)},
            watermark_sequence = coalesce(watermark_sequence, 0) + 1,
            last_run_id = {sql_string(RUN_ID)},
            error_message = NULL
        WHERE watermark_id = {sql_string(WATERMARK_ID)}
    """)


def fail_watermark(error_message):
    msg = str(error_message)[:4000]
    spark.sql(f"""
        UPDATE {WATERMARK_TBL}
        SET
            update_time = current_timestamp(),
            status = 'FAILED',
            flow_name = {sql_string(FLOW_NAME)},
            last_run_id = {sql_string(RUN_ID)},
            error_message = {sql_string(msg)}
        WHERE watermark_id = {sql_string(WATERMARK_ID)}
    """)


def notebook_exit(payload):
    text = json.dumps(payload, ensure_ascii=False, default=str)
    print("=" * 100)
    print(text)
    print("=" * 100)
    try:
        notebookutils.notebook.exit(text)
    except NameError:
        pass


def elapsed_seconds():
    return round(time.perf_counter() - RUN_STARTED_MONOTONIC, 3)



def step_timer_start():
    return time.perf_counter()


def log_step_done(stage, started_at, message, **kwargs):
    step_seconds = round(time.perf_counter() - started_at, 3)
    log_stage(stage, message, step_seconds=step_seconds, **kwargs)
    return step_seconds

def log_stage(stage, message="", **metrics):
    metric_text = " ".join(f"{k}={v}" for k, v in metrics.items())
    suffix = " ".join(x for x in [message, metric_text] if x)
    print(f"[POI_CHANGE][{stage}] elapsed={elapsed_seconds()}s {suffix}".rstrip())


def update_pipeline_duration():
    """Best-effort duration persistence; supports common numeric/string/interval schemas."""
    if not table_exists(PIPELINE_RUN_TBL):
        return
    schema = {f.name: f.dataType.simpleString().lower() for f in spark.table(PIPELINE_RUN_TBL).schema.fields}
    if "duration" not in schema:
        return
    dtype = schema["duration"]
    sec = elapsed_seconds()
    if any(x in dtype for x in ["int", "long", "bigint", "double", "float", "decimal"]):
        expr = str(sec)
    elif "string" in dtype:
        expr = sql_string(f"{sec:.3f}s")
    elif "interval" in dtype:
        expr = f"INTERVAL {sec} SECONDS"
    else:
        # Unknown physical type: keep the run successful and rely on started_at/ended_at.
        return
    spark.sql(f"""
        UPDATE {PIPELINE_RUN_TBL}
        SET duration = {expr}
        WHERE run_id = {sql_string(RUN_ID)}
    """)


def physical_type_sql(dtype):
    """
    Spark SQL datatype string suitable for CAST/TRY_CAST.
    """
    return dtype.simpleString()


def target_schema_map(target):
    return {f.name: f for f in spark.table(target).schema.fields}


def business_patch_columns(target):
    cols = columns(target)
    return [
        c for c in cols
        if c not in PROTECTED_PATCH_COLUMNS
    ]


# =============================================================================
# 2. PREFLIGHT
# =============================================================================

for tbl in [
    BRONZE_EVENT_TBL,
    REJECTED_TBL,
    WATERMARK_TBL,
    PIPELINE_RUN_TBL,
]:
    if not table_exists(tbl):
        raise RuntimeError(f"REQUIRED_TABLE_NOT_FOUND table={tbl}")

validate_columns(
    BRONZE_EVENT_TBL,
    REQUIRED_BRONZE_COLUMNS,
    "BRONZE_EVENT"
)
validate_columns(
    WATERMARK_TBL,
    REQUIRED_WATERMARK_COLUMNS,
    "WATERMARK"
)
validate_columns(
    PIPELINE_RUN_TBL,
    {"run_id", "flow_name", "status", "started_at"},
    "PIPELINE_RUN"
)

for entity, cfg in ENTITY_CONFIG.items():
    target = cfg["target"]

    if not table_exists(target):
        raise RuntimeError(
            f"GOLD_TARGET_NOT_FOUND entity={entity} table={target}"
        )

    validate_columns(
        target,
        {"poi_id"},
        f"TARGET_{entity}"
    )


bootstrap_watermark()

wm = watermark_row()
if wm is None:
    raise RuntimeError("WATERMARK_BOOTSTRAP_FAILED")

wm_dict = wm.asDict(recursive=True)

if (
    wm_dict.get("table_name") not in (None, "", WATERMARK_TABLE_NAME)
    or wm_dict.get("watermark_column")
       not in (None, "", WATERMARK_SOURCE_COLUMN)
):
    raise RuntimeError(
        "WATERMARK_CONTRACT_MISMATCH "
        f"watermark_id={WATERMARK_ID} "
        f"table_name={wm_dict.get('table_name')} "
        f"watermark_column={wm_dict.get('watermark_column')}"
    )

wm_value = wm_dict.get("watermark_value")
watermark_from = (
    str(wm_value)
    if wm_value is not None
    else INITIAL_WATERMARK
)


if str(watermark_from).startswith(str(INITIAL_WATERMARK)[:10]):
    print(
        f"[POI_CHANGE][INFO] watermark_from is bootstrap watermark={INITIAL_WATERMARK}. "
        "After a successful run, brz_watermark should advance to the fixed watermark_to."
    )


# =============================================================================
# 3. FREEZE UPPER BOUND - V10 FAST PATH
# =============================================================================
# Do not scan the full Bronze history for NULL ingest_timestamp on every run.
# ingest_timestamp validity is enforced on events that enter the processing
# window. A separate DQ/monitoring job should own historical NULL-cursor checks.
# =============================================================================

upper = spark.sql(f"""
    SELECT MAX(ingest_timestamp) AS wm_to
    FROM {BRONZE_EVENT_TBL}
    WHERE ingest_timestamp > CAST({sql_string(watermark_from)} AS TIMESTAMP)
""").first()["wm_to"]

watermark_to = str(upper) if upper is not None else watermark_from

window_start = spark.sql(f"""
    SELECT CAST({sql_string(watermark_from)} AS TIMESTAMP)
           - INTERVAL {LOOKBACK_MINUTES} MINUTES AS x
""").first()["x"]


# =============================================================================
# 4. LOAD FIXED RAW WINDOW
# =============================================================================

raw = (
    spark.table(BRONZE_EVENT_TBL)
    .filter(
        (F.col("ingest_timestamp") > F.lit(window_start))
        & (
            F.col("ingest_timestamp")
            <= F.to_timestamp(F.lit(watermark_to))
        )
    )
    .select(
        F.trim(F.col("eventId")).alias("event_id"),
        F.trim(F.col("poiId")).alias("poi_id"),
        F.lower(F.trim(F.col("entity"))).alias("entity"),
        F.col("version").cast("long").alias("version"),
        F.upper(F.trim(F.col("eventType"))).alias("event_type"),
        F.col("source").cast("string").alias("source"),
        F.col("changes").cast("string").alias("changes_json"),
        F.col("actor").cast("string").alias("actor"),
        F.col("occurredAt").cast("string").alias("occurred_at"),
        F.col("schemaVersion").cast("string").alias("schema_version"),
        F.col("ingest_timestamp"),
    )
)

raw = raw.persist()
raw_count = raw.count()
log_stage("RAW_WINDOW", "Fixed watermark window loaded", raw_event_count=raw_count, watermark_from=watermark_from, watermark_to=watermark_to)

insert_pipeline_run(
    watermark_from,
    watermark_to,
    raw_count
)
set_watermark_running()


# =============================================================================
# 5. NO-DATA PATH
# =============================================================================

if raw_count == 0:
    advance_watermark(watermark_to)
    update_pipeline_run("SUCCESS", close_run=True)
    update_pipeline_duration()
    raw.unpersist()
    log_stage("COMPLETE", "No new events", total_seconds=elapsed_seconds())

    notebook_exit({
        "status": "NO_DATA",
        "run_id": RUN_ID,
        "flow_name": FLOW_NAME,
        "watermark_from": watermark_from,
        "watermark_to": watermark_to,
        "raw_event_count": 0,
    })


# =============================================================================
# 6. WINDOW-LOCAL EVENT ID DEDUP - V10.4
# =============================================================================
# Persistent event-processed table is not used.
# V11: durable ordering is resolved from immutable Bronze history at change-key level.
# =============================================================================
candidate_raw = raw

w_event = Window.partitionBy("event_id").orderBy(
    F.col("ingest_timestamp").desc_nulls_last(),
    F.col("version").desc_nulls_last()
)

candidate = (
    candidate_raw
    .withColumn(
        "_event_rn",
        F.when(
            F.col("event_id").isNotNull() & (F.trim(F.col("event_id")) != ""),
            F.row_number().over(w_event)
        ).otherwise(F.lit(1))
    )
    .filter(F.col("_event_rn") == 1)
)


# =============================================================================
# 9. BASE VALIDATION
# =============================================================================

candidate = (
    candidate
    .withColumn(
        "_json_valid",
        F.get_json_object(
            F.col("changes_json"),
            "$"
        ).isNotNull()
    )
    .withColumn(
        "_change_keys",
        F.map_keys(
            F.from_json(
                F.col("changes_json"),
                "map<string,string>"
            )
        )
    )
)

base_error = (
    F.when(
        F.col("event_id").isNull()
        | (F.trim(F.col("event_id")) == ""),
        F.lit("MISSING_EVENT_ID")
    )
    .when(
        F.col("poi_id").isNull()
        | (F.trim(F.col("poi_id")) == ""),
        F.lit("MISSING_POI_ID")
    )
    .when(
        F.col("version").isNull(),
        F.lit("MISSING_VERSION")
    )
    .when(
        ~F.col("entity").isin(*ENTITY_CONFIG.keys()),
        F.lit("UNSUPPORTED_ENTITY")
    )
    .when(
        F.col("event_type").isNull()
        | (~F.col("event_type").isin(*SUPPORTED_EVENT_TYPES)),
        F.lit("UNSUPPORTED_EVENT_TYPE")
    )
    .when(
        F.col("changes_json").isNull()
        | (F.trim(F.col("changes_json")) == ""),
        F.lit("EMPTY_CHANGES")
    )
    .when(
        ~F.col("_json_valid"),
        F.lit("INVALID_CHANGES_JSON")
    )
)

candidate = candidate.withColumn(
    "_error",
    base_error
)


# =============================================================================
# 10. ENTITY-SPECIFIC PATCH VALIDATION - V10 FAST PATH
# =============================================================================
# Validate the JSON contract against target schema, but do NOT join each Gold
# target merely to check poi_id existence. Gold MERGE is MATCHED-only.
#
# Per-field cast/null checks remain for data safety. They are Catalyst
# expressions over the small candidate set and do not create one Spark action
# per field.
# =============================================================================

validated_parts = []

for entity, cfg in ENTITY_CONFIG.items():
    target = cfg["target"]
    schema_map = target_schema_map(target)
    part = candidate.filter(F.col("entity") == entity)

    allowed_cols = [
        c for c in business_patch_columns(target)
        if c in schema_map
    ]

    allowed_array = F.array(*[F.lit(c) for c in allowed_cols])

    part = part.withColumn(
        "_unsupported_keys",
        F.array_except(
            F.coalesce(F.col("_change_keys"), F.array().cast("array<string>")),
            allowed_array
        )
    )

    part = part.withColumn(
        "_error",
        F.when(F.col("_error").isNotNull(), F.col("_error"))
         .when(
             F.size(F.col("_unsupported_keys")) > 0,
             F.concat(
                 F.lit("UNSUPPORTED_PATCH_FIELDS:"),
                 F.concat_ws(",", F.col("_unsupported_keys"))
             )
         )
    )

    for c in allowed_cols:
        field = schema_map[c]
        dtype_sql = physical_type_sql(field.dataType)
        has_key = F.array_contains(
            F.coalesce(F.col("_change_keys"), F.array().cast("array<string>")), c
        )
        raw_value = F.get_json_object(F.col("changes_json"), f"$.{c}")
        cast_value = F.expr(
            f"try_cast(get_json_object(changes_json, '$.{c}') AS {dtype_sql})"
        )

        part = part.withColumn(
            "_error",
            F.when(F.col("_error").isNotNull(), F.col("_error"))
             .when(
                 has_key & raw_value.isNull() & F.lit(not field.nullable),
                 F.lit(f"NULL_NOT_ALLOWED:{c}")
             )
             .when(
                 has_key & raw_value.isNotNull() & cast_value.isNull(),
                 F.lit(f"INVALID_CAST:{c}:{dtype_sql}")
             )
        )

    validated_parts.append(part)

unsupported = candidate.filter(~F.col("entity").isin(*ENTITY_CONFIG.keys()))
validated = unsupported
for p in validated_parts:
    validated = validated.unionByName(p, allowMissingColumns=True)


# =============================================================================
# 11. V11 EVENT CLASSIFICATION - NO ENTITY VERSION CONTROL TABLE
# =============================================================================
# Valid events remain eligible for key-level resolution. Durable ordering is
# enforced later in Section 12 against Bronze history at grain:
#     (poi_id, entity, change_key)
# This is required because different versions may change different keys.
# An older event is NOT stale as a whole when it still owns the latest value
# for at least one key.
# =============================================================================

validated = validated.withColumn("_is_stale", F.lit(False))

ready = validated.filter(F.col("_error").isNull())
rejected = validated.filter(F.col("_error").isNotNull())
stale = validated.filter(F.lit(False))

validated = validated.persist()

_accounting = validated.agg(
    F.count(F.lit(1)).alias("candidate_count"),
    F.sum(F.when(F.col("_error").isNull(), 1).otherwise(0)).alias("ready_count"),
    F.sum(F.when(F.col("_error").isNotNull(), 1).otherwise(0)).alias("rejected_count"),
).first()

candidate_count = int(_accounting["candidate_count"] or 0)
ready_count = int(_accounting["ready_count"] or 0)
rejected_count = int(_accounting["rejected_count"] or 0)
stale_count = 0

ready = ready.persist()
if ready_count > 0:
    ready.count()

log_stage(
    "VALIDATION",
    "Event validation completed; key-level history resolution deferred to Gold patch",
    candidate_event_count=candidate_count,
    ready_event_count=ready_count,
    rejected_event_count=rejected_count,
    stale_event_count=stale_count,
)


def execute_delta_merge_with_retry(sql_text, label, max_attempts=4):
    """
    Execute one Delta mutation with bounded optimistic-concurrency retry.

    V10.9 observability:
      - always logs attempt START / DONE / ERROR
      - makes a driver/catalog/Delta-commit stall visible immediately
      - retries only known optimistic-concurrency failures
    """
    delays = [2, 5, 10]

    for attempt in range(1, max_attempts + 1):
        attempt_started = time.perf_counter()
        print(
            f"[POI_CHANGE][DELTA_MERGE_START] "
            f"label={label} attempt={attempt}"
        )

        try:
            spark.sql(sql_text)

            attempt_seconds = round(
                time.perf_counter() - attempt_started, 3
            )
            print(
                f"[POI_CHANGE][DELTA_MERGE_DONE] "
                f"label={label} attempt={attempt} "
                f"step_seconds={attempt_seconds}"
            )

            if attempt > 1:
                print(
                    f"[POI_CHANGE][RETRY_OK] "
                    f"label={label} attempt={attempt}"
                )
            return

        except Exception as exc:
            attempt_seconds = round(
                time.perf_counter() - attempt_started, 3
            )
            msg = str(exc)

            print(
                f"[POI_CHANGE][DELTA_MERGE_ERROR] "
                f"label={label} attempt={attempt} "
                f"step_seconds={attempt_seconds} "
                f"error={msg[:1000]}"
            )

            retryable = any(token in msg for token in [
                "DELTA_CONCURRENT_APPEND",
                "ConcurrentAppendException",
                "ConcurrentWriteException",
                "ConcurrentDeleteReadException",
                "DELTA_CONCURRENT",
            ])

            if (not retryable) or attempt >= max_attempts:
                raise

            delay = delays[min(attempt - 1, len(delays) - 1)]
            print(
                f"[POI_CHANGE][RETRY] label={label} attempt={attempt} "
                f"sleep_seconds={delay} reason=DELTA_CONCURRENT_WRITE"
            )
            time.sleep(delay)




# =============================================================================
# 11B. PERSIST REJECTED EVENTS TO BRONZE DLQ - V10.7
# =============================================================================
# Existing target:
#   lh_vv_bronze.dbo.poi_change_event_rejected
#
# IMPORTANT V10.7 FIX:
#   Bronze `changes` is normalized to `changes_json` in Section 4.
#   Validation reason is stored in `_error`.
#
# Therefore DLQ mapping MUST be:
#   target.changes      <- rejected.changes_json
#   target.error_reason <- rejected._error
#
# V10.6 looked for source columns named `changes` and `reject_reason`, which
# do not exist in the rejected DataFrame. That is why both target columns were
# NULL even though the event was rejected.
#
# DLQ remains WRITE-ONLY from this processor and is never joined into the hot
# path. MERGE by event_id makes lookback/retry idempotent and also repairs
# existing rows whose changes/error_reason were previously NULL.
# =============================================================================

_reject_dlq_started = step_timer_start()

log_stage(
    "REJECT_DLQ_START",
    "Persisting rejected events with original changes and validation reason",
    rejected_event_count=rejected_count,
)

if rejected_count > 0:
    _target_schema = spark.table(REJECTED_TBL).schema
    _target_cols = [f.name for f in _target_schema.fields]
    _source_cols = set(rejected.columns)

    # Explicit mapping from existing target schema to the ACTUAL columns in
    # the rejected DataFrame.
    _source_mapping = {
        "event_id": "event_id",
        "eventId": "event_id",

        "poi_id": "poi_id",
        "poiId": "poi_id",

        "entity": "entity",

        "event_type": "event_type",
        "eventType": "event_type",

        "version": "version",

        # Critical V10.7 fixes:
        "changes": "changes_json",
        "changes_json": "changes_json",
        "error_reason": "_error",
        "reject_reason": "_error",
        "reason": "_error",

        "source": "source",
        "actor": "actor",

        "occurred_at": "occurred_at",
        "occurredAt": "occurred_at",

        "schema_version": "schema_version",
        "schemaVersion": "schema_version",

        "ingest_timestamp": "ingest_timestamp",
    }

    _select_exprs = []
    _mapped_cols = []

    for _tc in _target_cols:
        _src = _source_mapping.get(_tc)

        # Same-name fallback for any additional existing target columns.
        if _src is None and _tc in _source_cols:
            _src = _tc

        if _src is not None and _src in _source_cols:
            _select_exprs.append(
                F.col(_src).cast(
                    next(f.dataType for f in _target_schema.fields if f.name == _tc)
                ).alias(_tc)
            )
            _mapped_cols.append(_tc)

        elif _tc == "run_id":
            _select_exprs.append(F.lit(RUN_ID).alias(_tc))
            _mapped_cols.append(_tc)

        elif _tc in ("rejected_at", "created_at", "updated_at"):
            _select_exprs.append(F.current_timestamp().alias(_tc))
            _mapped_cols.append(_tc)

    if not _select_exprs:
        raise RuntimeError(
            f"REJECT_DLQ_NO_MAPPABLE_COLUMNS table={REJECTED_TBL}"
        )

    # Hard safety checks for the two fields that motivated V10.7.
    if "changes" in _target_cols and "changes" not in _mapped_cols:
        raise RuntimeError(
            "REJECT_DLQ_MAPPING_ERROR: target.changes exists but "
            "source changes_json was not mapped"
        )

    if "error_reason" in _target_cols and "error_reason" not in _mapped_cols:
        raise RuntimeError(
            "REJECT_DLQ_MAPPING_ERROR: target.error_reason exists but "
            "source _error was not mapped"
        )

    _reject_write = rejected.select(*_select_exprs)

    # event_id should be unique inside this batch before MERGE. The Bronze
    # window has already been deduplicated, but keep this guard local to DLQ.
    _event_key = (
        "event_id" if "event_id" in _mapped_cols
        else ("eventId" if "eventId" in _mapped_cols else None)
    )

    if _event_key:
        _reject_write = _reject_write.dropDuplicates([_event_key])

    _reject_view = f"_reject_dlq_{uuid.uuid4().hex}"
    _reject_write.createOrReplaceTempView(_reject_view)

    if _event_key:
        _update_cols = [c for c in _mapped_cols if c != _event_key]

        _matched_clause = ""
        if _update_cols:
            _matched_clause = (
                "WHEN MATCHED THEN UPDATE SET\n                "
                + ",\n                ".join(
                    [f"t.`{c}` = s.`{c}`" for c in _update_cols]
                )
            )

        _insert_cols = ", ".join(
            f"`{c}`" for c in _mapped_cols
        )
        _insert_vals = ", ".join(
            f"s.`{c}`" for c in _mapped_cols
        )

        execute_delta_merge_with_retry(f"""
            MERGE INTO {REJECTED_TBL} t
            USING {_reject_view} s
            ON t.`{_event_key}` = s.`{_event_key}`

            {_matched_clause}

            WHEN NOT MATCHED THEN INSERT (
                {_insert_cols}
            )
            VALUES (
                {_insert_vals}
            )
        """, "REJECT_DLQ_MERGE")

    else:
        # Fallback only when the pre-created target genuinely has no event key.
        _reject_write.write.format("delta").mode("append").saveAsTable(
            REJECTED_TBL
        )

    spark.catalog.dropTempView(_reject_view)

log_step_done(
    "REJECT_DLQ_DONE",
    _reject_dlq_started,
    "Rejected event persistence completed",
    rejected_event_count=rejected_count,
    changes_source="changes_json",
    reason_source="_error",
)


# =============================================================================
# 12. PATCH CURRENT GOLD BUSINESS TABLES - V9 OPTIMIZED
# =============================================================================
# One Window per entity over (poi_id, patch_key), replacing one Window + Join
# per physical target column. Missing key retains current value; explicit null
# sets NULL; latest version/event containing each field wins.
# =============================================================================

patched_counts = {}

for entity, cfg in ENTITY_CONFIG.items():
    _gold_entity_started = step_timer_start()
    log_stage(
        f"GOLD_{entity.upper()}_START",
        "Starting Gold entity patch",
        entity=entity,
    )
    target = cfg["target"]
    schema_map = target_schema_map(target)
    entity_ready = ready.filter(F.col("entity") == entity).persist()

    affected_count = int(
        entity_ready.agg(F.countDistinct("poi_id").alias("c")).first()["c"] or 0
    )
    if affected_count == 0:
        patched_counts[entity] = 0
        entity_ready.unpersist()
        continue

    allowed_cols = [
        c for c in business_patch_columns(target)
        if c in schema_map and c != "updated_at"
    ]

    # -------------------------------------------------------------------------
    # V11 KEY-LEVEL HISTORY RESOLUTION
    # -------------------------------------------------------------------------
    # 1) Determine only keys touched by the current valid batch.
    # 2) Read Bronze history only for affected POIs of this entity.
    # 3) Keep only fully contract-valid historical events.
    # 4) Explode changes and restrict to currently affected keys.
    # 5) Latest (version DESC, event_id DESC) value per (poi_id, patch_key) wins.
    #
    # This preserves non-overlapping keys across versions:
    #   v10 {A,B}, v11 {A}, v12 {C} => A@v11, B@v10, C@v12
    # Explicit JSON null remains a real patch and is not treated as missing.
    # -------------------------------------------------------------------------
    current_kv = (
        entity_ready
        .select(
            "poi_id", "version", "event_id",
            F.explode(
                F.from_json(F.col("changes_json"), "map<string,string>")
            ).alias("patch_key", "patch_raw")
        )
        .filter(F.col("patch_key").isin(*allowed_cols))
    )

    affected_keys = current_kv.select("poi_id", "patch_key").distinct().persist()
    affected_pois = affected_keys.select("poi_id").distinct()

    history_events = (
        spark.table(BRONZE_EVENT_TBL)
        .select(
            F.trim(F.col("eventId")).alias("event_id"),
            F.trim(F.col("poiId")).alias("poi_id"),
            F.lower(F.trim(F.col("entity"))).alias("entity"),
            F.col("version").cast("long").alias("version"),
            F.upper(F.trim(F.col("eventType"))).alias("event_type"),
            F.col("changes").cast("string").alias("changes_json"),
        )
        .filter(F.col("entity") == entity)
        .filter(F.col("version").isNotNull())
        .filter(F.col("event_id").isNotNull() & (F.col("event_id") != ""))
        .filter(F.col("event_type").isin(*SUPPORTED_EVENT_TYPES))
        .join(F.broadcast(affected_pois), ["poi_id"], "inner")
        .withColumn(
            "_history_change_keys",
            F.map_keys(F.from_json(F.col("changes_json"), "map<string,string>"))
        )
        .filter(F.col("_history_change_keys").isNotNull())
        .withColumn(
            "_history_unsupported_keys",
            F.array_except(
                F.coalesce(F.col("_history_change_keys"), F.array().cast("array<string>")),
                F.array(*[F.lit(c) for c in allowed_cols])
            )
        )
        # Match current validation semantics: an event containing any
        # unsupported patch field is rejected as a whole.
        .filter(F.size(F.col("_history_unsupported_keys")) == 0)
    )

    history_kv = (
        history_events
        .select(
            "poi_id", "version", "event_id",
            F.explode(
                F.from_json(F.col("changes_json"), "map<string,string>")
            ).alias("patch_key", "patch_raw")
        )
        .join(F.broadcast(affected_keys), ["poi_id", "patch_key"], "inner")
    )

    # Historical events that would have failed the current target-type/null
    # contract must not suppress an older valid value.
    _valid_history_key = F.lit(False)
    for c in allowed_cols:
        field = schema_map[c]
        dtype_sql = physical_type_sql(field.dataType)
        cast_ok = F.expr(f"try_cast(patch_raw AS {dtype_sql})").isNotNull()
        null_ok = F.col("patch_raw").isNull() & F.lit(bool(field.nullable))
        _valid_history_key = _valid_history_key | (
            (F.col("patch_key") == F.lit(c)) & (cast_ok | null_ok)
        )

    history_kv = history_kv.filter(_valid_history_key)

    w_patch = Window.partitionBy("poi_id", "patch_key").orderBy(
        F.col("version").desc(),
        F.col("event_id").desc_nulls_last()
    )

    latest_kv = (
        history_kv
        .withColumn("_rn_patch", F.row_number().over(w_patch))
        .filter(F.col("_rn_patch") == 1)
        .drop("_rn_patch")
        .persist()
    )

    latest_key_count = latest_kv.count()
    log_stage(
        f"GOLD_{entity.upper()}_KEY_HISTORY",
        "Latest Bronze value resolved per affected change key",
        entity=entity,
        affected_poi_count=affected_count,
        resolved_key_count=latest_key_count,
    )

    agg_exprs = []
    for c in allowed_cols:
        dtype_sql = physical_type_sql(schema_map[c].dataType)
        agg_exprs.append(
            F.max(F.when(F.col("patch_key") == c, F.lit(True))).alias(f"_has_{c}")
        )
        agg_exprs.append(
            F.max(
                F.when(
                    F.col("patch_key") == c,
                    F.expr(f"try_cast(patch_raw AS {dtype_sql})")
                )
            ).alias(f"_patch_{c}")
        )

    patch_base = (
        latest_kv.groupBy("poi_id").agg(*agg_exprs)
        if agg_exprs
        else entity_ready.select("poi_id").distinct()
    )

    patch_view = f"_patch_v9_{entity}_{uuid.uuid4().hex}"
    patch_base.createOrReplaceTempView(patch_view)

    assignments = [
        f"t.{qcol(c)} = CASE "
        f"WHEN coalesce(s.{qcol(f'_has_{c}')}, false) "
        f"THEN s.{qcol(f'_patch_{c}')} ELSE t.{qcol(c)} END"
        for c in allowed_cols
    ]
    if "updated_at" in schema_map:
        assignments.append("t.updated_at = current_timestamp()")

    if assignments:
        execute_delta_merge_with_retry(f"""
            MERGE INTO {target} t
            USING {patch_view} s
            ON t.poi_id = s.poi_id
            WHEN MATCHED THEN UPDATE SET
                {", ".join(assignments)}
        """, f"GOLD_MERGE:{entity}")

    patched_counts[entity] = affected_count
    log_step_done(
        f"GOLD_{entity.upper()}_DONE",
        _gold_entity_started,
        "Gold entity patch completed",
        entity=entity,
        patched_count=affected_count,
    )
    spark.catalog.dropTempView(patch_view)
    latest_kv.unpersist()
    affected_keys.unpersist()
    entity_ready.unpersist()

log_stage(
    "GOLD_PATCH",
    "Gold current-state patch completed",
    **{f"patched_{k}": v for k, v in patched_counts.items()}
)


# =============================================================================
# 12B. V10.2 - ONE UNIFIED AI REFRESH / ONE MERGE
# =============================================================================
# Previous versions MERGEd ai_enriched_poi once per changed entity. On Fabric,
# each Delta MERGE can scan/rewrite target files, so 2-3 MERGEs were expensive
# even when only a few POIs changed.
#
# V10.2:
#   ready events -> distinct affected poi_id
#                -> read CURRENT Gold master/detail/enrichment state once
#                -> build one schema-aware AI refresh source
#                -> ONE MERGE into ai_enriched_poi
#
# NB00-NB60 remains the owner of initial AI materialization. Change Flow only
# updates existing (poi_id, lang) rows.
# =============================================================================

AI_ENRICHED_TBL = f"{GOLD_DB}.ai_enriched_poi"

if not table_exists(AI_ENRICHED_TBL):
    raise RuntimeError(f"AI_ENRICHED_TARGET_NOT_FOUND table={AI_ENRICHED_TBL}")

validate_columns(AI_ENRICHED_TBL, {"poi_id", "lang"}, "AI_ENRICHED_TARGET")

_ai_target_schema = target_schema_map(AI_ENRICHED_TBL)
_ai_target_cols = set(_ai_target_schema.keys())

master_tbl = ENTITY_CONFIG["poi_master"]["target"]
detail_tbl = ENTITY_CONFIG["poi_detail"]["target"]
enrich_tbl = ENTITY_CONFIG["poi_enrichment_display"]["target"]

master_cols = column_set(master_tbl)
detail_cols = column_set(detail_tbl)
enrich_cols = column_set(enrich_tbl)

_ai_affected = ready.select("poi_id").distinct().persist()
_ai_affected_count = int(_ai_affected.agg(F.count("*").alias("c")).first()["c"] or 0)

_ai_sync_started = step_timer_start()
log_stage(
    "AI_SYNC_START",
    "Starting ai_enriched_poi incremental sync",
)

ai_enriched_updated_counts = {
    "affected_poi_count": _ai_affected_count,
    "unified_merge_count": 0,
}

if _ai_affected_count > 0:
    aff_view = f"_ai_aff_v102_{uuid.uuid4().hex}"
    _ai_affected.createOrReplaceTempView(aff_view)

    protected_ai = {
        "poi_id", "lang", "created_at",
        "enrichment_id", "enrichment_job_id", "enrichment_status",
        "validation_status", "model_name", "source_updated_at", "enriched_at",
        "ref_cost_method", "ref_cost_confidence", "ref_reasoning_short",
    }

    mapped = {}

    # Master owns same-name canonical POI fields first.
    for c in sorted((master_cols & _ai_target_cols) - protected_ai):
        mapped[c] = f"m.{qcol(c)}"

    # Detail may supply same-name serving fields not already owned by master.
    for c in sorted((detail_cols & _ai_target_cols) - protected_ai):
        mapped.setdefault(c, f"d.{qcol(c)}")

    # Enrichment direct same-name fields are lowest direct-map priority.
    for c in sorted((enrich_cols & _ai_target_cols) - protected_ai):
        mapped.setdefault(c, f"e.{qcol(c)}")

    # business_category is copied exactly from current poi_master.
    if "business_category" in _ai_target_cols and "business_category" in master_cols:
        mapped["business_category"] = "m.`business_category`"

    # Detail JSON -> AI serving fields.
    if "poi_address_json" in detail_cols:
        mapped.update({
            k: v for k, v in {
                "address_line_1": "get_json_object(d.`poi_address_json`, '$.address_line1')",
                "address_line_2": "get_json_object(d.`poi_address_json`, '$.address_line2')",
                "ward": "get_json_object(d.`poi_address_json`, '$.ward')",
                "district": "get_json_object(d.`poi_address_json`, '$.district')",
                "city": "get_json_object(d.`poi_address_json`, '$.city')",
                "province_name": "coalesce(get_json_object(d.`poi_address_json`, '$.state_province'), get_json_object(d.`poi_address_json`, '$.city'))",
                "country_code": "get_json_object(d.`poi_address_json`, '$.country_code')",
                "lat": "get_json_object(d.`poi_address_json`, '$.lat')",
                "lng": "get_json_object(d.`poi_address_json`, '$.lng')",
            }.items() if k in _ai_target_cols
        })

    if "poi_contact_json" in detail_cols:
        for c in ["phone", "email", "website_url", "booking_url",
                  "facebook_url", "instagram_url", "tiktok_url"]:
            if c in _ai_target_cols:
                mapped[c] = f"get_json_object(d.`poi_contact_json`, '$.{c}')"

    if "poi_content_json" in detail_cols:
        content_map = {
            "description_short": "get_json_object(d.`poi_content_json`, '$.description_short')",
            "description_long": "get_json_object(d.`poi_content_json`, '$.description_long')",
            "description": "get_json_object(d.`poi_content_json`, '$.description_short')",
            "story": "get_json_object(d.`poi_content_json`, '$.description_long')",
            "tags_text": "get_json_object(d.`poi_content_json`, '$.tags_text')",
        }
        for c, expr in content_map.items():
            if c in _ai_target_cols:
                mapped[c] = expr

    if "poi_opening_hours_json" in detail_cols:
        hours_map = {
            "timezone": "get_json_object(d.`poi_opening_hours_json`, '$.timezone')",
            "is_24_7": "get_json_object(d.`poi_opening_hours_json`, '$.is_24_7')",
            "open_now": "get_json_object(d.`poi_opening_hours_json`, '$.open_now')",
            "operating_hours": "coalesce(get_json_object(d.`poi_opening_hours_json`, '$.operating_hours'), get_json_object(d.`poi_opening_hours_json`, '$.periods_json'))",
            "holiday_exceptions": "get_json_object(d.`poi_opening_hours_json`, '$.holiday_exceptions')",
        }
        for c, expr in hours_map.items():
            if c in _ai_target_cols:
                mapped[c] = expr

    if "poi_policy_json" in detail_cols and "policy_json" in _ai_target_cols:
        mapped["policy_json"] = "d.`poi_policy_json`"

    if "amenity_json" in detail_cols:
        amenity_candidates = [
            "takeout", "delivery", "dine_in", "curbside_pickup", "reservable",
            "serves_breakfast", "serves_lunch", "serves_dinner", "serves_beer",
            "serves_wine", "serves_brunch", "serves_vegetarian_food",
            "outdoor_seating", "live_music", "menu_for_children",
            "serves_cocktails", "serves_dessert", "serves_coffee",
            "good_for_children", "allows_dogs", "has_restroom", "good_for_groups",
            "good_for_watching_sports", "pure_service_area_business", "parking",
            "ev_charge", "payment_options_accepts_credit_cards",
            "payment_options_accepts_debit_cards", "payment_options_accepts_nfc",
            "parking_options_free_parking_lot", "parking_options_paid_parking_lot",
            "parking_options_free_street_parking", "parking_options_paid_street_parking",
            "parking_options_valet_parking", "parking_options_free_garage_parking",
            "parking_options_paid_garage_parking",
            "accessibility_options_wheelchair_accessible_parking",
            "accessibility_options_wheelchair_accessible_entrance",
            "accessibility_options_wheelchair_accessible_restroom",
            "accessibility_options_wheelchair_accessible_seating",
            "has_swimming_pool", "has_gym", "has_spa", "has_sauna", "has_garden",
            "has_bar_on_site", "has_restaurant_on_site", "has_coworking_space",
            "general_amenities",
        ]
        for c in amenity_candidates:
            if c in _ai_target_cols:
                mapped[c] = f"get_json_object(d.`amenity_json`, '$.{c}')"

    enrichment_alias_candidates = {
        "ref_description": ["description", "ref_description"],
        "ref_time_to_spend_minutes": ["time_to_spend_minutes", "ref_time_to_spend_minutes"],
        "ref_best_time_type": ["best_time_type", "ref_best_time_type"],
        "ref_best_time_from_month": ["best_time_from_month", "ref_best_time_from_month"],
        "ref_best_time_to_month": ["best_time_to_month", "ref_best_time_to_month"],
        "ref_cost_vnd": ["cost_vnd", "ref_cost_vnd"],
        "ref_experiences": ["experiences_json", "experiences", "ref_experiences"],
        "ref_review_summary": ["review_summary", "ref_review_summary"],
    }
    for target_col, candidates in enrichment_alias_candidates.items():
        if target_col not in _ai_target_cols:
            continue
        for source_col in candidates:
            if source_col in enrich_cols:
                mapped[target_col] = f"e.{qcol(source_col)}"
                break

    # updated_at is system-managed and set once.
    mapped.pop("updated_at", None)

    select_exprs = ["m.poi_id AS poi_id", "m.lang AS lang"]
    for target_col, expr_sql in mapped.items():
        dtype_sql = physical_type_sql(_ai_target_schema[target_col].dataType)
        select_exprs.append(
            f"try_cast(({expr_sql}) AS {dtype_sql}) AS {qcol(target_col)}"
        )
    if "updated_at" in _ai_target_cols:
        select_exprs.append("current_timestamp() AS `updated_at`")

    ai_src_view = f"_ai_src_v102_{uuid.uuid4().hex}"
    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW {ai_src_view} AS
        SELECT {", ".join(select_exprs)}
        FROM {master_tbl} m
        INNER JOIN {aff_view} x
            ON x.poi_id = m.poi_id
        LEFT JOIN {detail_tbl} d
            ON d.poi_id = m.poi_id
        LEFT JOIN {enrich_tbl} e
            ON e.poi_id = m.poi_id
        WHERE m.poi_id IS NOT NULL
          AND m.lang IS NOT NULL
    """)

    update_cols = list(mapped.keys())
    if "updated_at" in _ai_target_cols:
        update_cols.append("updated_at")

    if update_cols:
        update_sql = ",\n                ".join(
            f"t.{qcol(c)} = s.{qcol(c)}" for c in update_cols
        )

        _ai_delays = [2, 5, 10]
        for _ai_attempt in range(1, 5):
            try:
                # Recreate the source view from CURRENT Gold state for every
                # retry. This avoids retrying against a stale Gold snapshot.
                spark.sql(f"""
                    CREATE OR REPLACE TEMP VIEW {ai_src_view} AS
                    SELECT {", ".join(select_exprs)}
                    FROM {master_tbl} m
                    INNER JOIN {aff_view} x
                        ON x.poi_id = m.poi_id
                    LEFT JOIN {detail_tbl} d
                        ON d.poi_id = m.poi_id
                    LEFT JOIN {enrich_tbl} e
                        ON e.poi_id = m.poi_id
                    WHERE m.poi_id IS NOT NULL
                      AND m.lang IS NOT NULL
                """)

                spark.sql(f"""
                    MERGE INTO {AI_ENRICHED_TBL} t
                    USING {ai_src_view} s
                    ON t.poi_id = s.poi_id
                   AND t.lang = s.lang
                    WHEN MATCHED THEN UPDATE SET
                        {update_sql}
                """)

                if _ai_attempt > 1:
                    print(
                        f"[POI_CHANGE][RETRY_OK] label=AI_ENRICHED_MERGE "
                        f"attempt={_ai_attempt}"
                    )
                break

            except Exception as _ai_exc:
                _ai_msg = str(_ai_exc)
                _ai_retryable = any(token in _ai_msg for token in [
                    "DELTA_CONCURRENT_APPEND",
                    "ConcurrentAppendException",
                    "ConcurrentWriteException",
                    "ConcurrentDeleteReadException",
                    "DELTA_CONCURRENT",
                ])
                if (not _ai_retryable) or _ai_attempt >= 4:
                    raise
                _ai_delay = _ai_delays[min(_ai_attempt - 1, len(_ai_delays) - 1)]
                print(
                    f"[POI_CHANGE][RETRY] label=AI_ENRICHED_MERGE "
                    f"attempt={_ai_attempt} sleep_seconds={_ai_delay} "
                    f"reason=DELTA_CONCURRENT_WRITE"
                )
                time.sleep(_ai_delay)

    ai_enriched_updated_counts["unified_merge_count"] = _ai_affected_count

    spark.catalog.dropTempView(ai_src_view)
    spark.catalog.dropTempView(aff_view)

_ai_affected.unpersist()

print("AI enriched unified incremental sync:", ai_enriched_updated_counts)
log_step_done(
    "AI_SYNC_DONE",
    _ai_sync_started,
    "ONE ai_enriched_poi MERGE completed",
    **ai_enriched_updated_counts
)


# =============================================================================
# 13. EVENT STATUS - V10.4
# =============================================================================
# No persistent event-level MERGE. Counts remain in run/notebook logs.
# =============================================================================
processed_count = ready_count
rejected_count_persisted = rejected_count
stale_count_persisted = stale_count

log_stage(
    "EVENT_STATUS",
    "Persistent event-level write disabled",
    processed=processed_count,
    rejected=rejected_count_persisted,
    stale=stale_count_persisted,
)


# =============================================================================
# 14. V11 - NO poi_change_entity_version WRITE
# =============================================================================
# Durable ordering has already been resolved from Bronze history at key level
# before each Gold MERGE. There is intentionally no Delta control-table MERGE
# here. This removes the session-blocking ENTITY_VERSION_MERGE hot path.
# =============================================================================

log_stage(
    "KEY_LEVEL_ORDERING_DONE",
    "Durable ordering resolved from Bronze history; no entity-version table used",
    ready_event_count=ready_count,
)


# =============================================================================
# 15. ACCOUNTING INVARIANT
# =============================================================================

logical_total = (
    ready_count
    + rejected_count
    + stale_count
)

if candidate_count != logical_total:
    raise RuntimeError(
        f"EVENT_ACCOUNTING_MISMATCH "
        f"candidate={candidate_count} "
        f"ready={ready_count} "
        f"rejected={rejected_count} "
        f"stale={stale_count}"
    )

log_stage(
    "ACCOUNTING",
    "Batch accounting invariant passed",
    candidate=candidate_count,
    ready=ready_count,
    rejected=rejected_count,
    stale=stale_count,
)


# =============================================================================
# 16. COMMIT WATERMARK + PIPELINE RUN + CLEANUP + EXIT
# =============================================================================

_watermark_started = step_timer_start()

log_stage(
    "WATERMARK_START",
    "Advancing successful batch watermark",
    watermark_from=watermark_from,
    watermark_to=watermark_to,
)

advance_watermark(
    watermark_to
)

log_step_done(
    "WATERMARK_DONE",
    _watermark_started,
    "Batch watermark advanced",
    watermark_value=watermark_to,
)

update_pipeline_run(
    "SUCCESS",
    close_run=True
)

log_stage(
    "PIPELINE_RUN_DONE",
    "brz_pipeline_run updated",
    status="SUCCESS",
    run_id=RUN_ID,
)

update_pipeline_duration()

# Release persisted DataFrames before notebook exit.
for _df_name in ["ready", "validated", "raw"]:
    _df = globals().get(_df_name)
    if _df is not None:
        try:
            _df.unpersist()
        except Exception:
            pass

total_processing_seconds = elapsed_seconds()

log_stage(
    "COMPLETE",
    "POI change batch completed successfully",
    status="SUCCESS",
    total_seconds=total_processing_seconds,
    raw_event_count=raw_count,
    candidate_event_count=candidate_count,
    processed_count=ready_count,
    rejected_count=rejected_count,
    stale_count=stale_count,
    watermark_value=watermark_to,
)

notebook_exit({
    "status": "SUCCESS",
    "run_id": RUN_ID,
    "flow_name": FLOW_NAME,
    "total_processing_seconds": total_processing_seconds,
    "watermark_id": WATERMARK_ID,
    "watermark_from": watermark_from,
    "watermark_to": watermark_to,
    "raw_event_count": raw_count,
    "candidate_event_count": candidate_count,
    "processed_count": ready_count,
    "rejected_count": rejected_count,
    "stale_count": stale_count,
    "patched_poi_counts": patched_counts,
    "ai_enriched_updated_counts": ai_enriched_updated_counts,
    "rejected_event_table": REJECTED_TBL,
    "business_tables_updated": [
        "poi_master",
        "poi_detail",
        "poi_enrichment_display",
        "ai_enriched_poi"
    ],
    "next_step": "CHANGE_FLOW_COMPLETE_V10_8_PERFORMANCE_OBSERVABILITY",
})


# =============================================================================
# 17. FAILURE HANDLING / OBSERVABILITY NOTE
# =============================================================================
#
# Production recommendation: keep the Fabric activity failure visible and ensure
# orchestration marks the run FAILED if the notebook raises. The helper functions
# fail_watermark() and update_pipeline_run() are available for an outer wrapper.
# If notebook-local failure persistence is required, wrap Sections 3-16 in:
#
#     try:
#         ...
#     except Exception as e:
#         fail_watermark(e)
#         update_pipeline_run("FAILED", e, close_run=True)
#         raise
#
# This file keeps all mutation logic explicit and leaves Fabric activity failure
# visible. For production, the recommended version is the wrapped variant.
# =============================================================================


StatementMeta(, 58944883-3e4f-488e-9981-9a6262b82e18, 3, Finished, Available, Finished, False)

[POI_CHANGE][RAW_WINDOW] elapsed=50.411s Fixed watermark window loaded raw_event_count=6 watermark_from=2026-09-16 10:06:59.595000 watermark_to=2026-09-16 11:39:55.189000
[POI_CHANGE][VALIDATION] elapsed=90.862s Event validation completed; key-level history resolution deferred to Gold patch candidate_event_count=6 ready_event_count=6 rejected_event_count=0 stale_event_count=0
[POI_CHANGE][REJECT_DLQ_START] elapsed=90.862s Persisting rejected events with original changes and validation reason rejected_event_count=0
[POI_CHANGE][REJECT_DLQ_DONE] elapsed=90.863s Rejected event persistence completed step_seconds=0.001 rejected_event_count=0 changes_source=changes_json reason_source=_error
[POI_CHANGE][GOLD_POI_MASTER_START] elapsed=90.864s Starting Gold entity patch entity=poi_master
[POI_CHANGE][GOLD_POI_MASTER_KEY_HISTORY] elapsed=108.182s Latest Bronze value resolved per affected change key entity=poi_master affected_poi_count=3 resolved_key_count=15
[POI_CHANGE][DELTA_MERGE_START] labe